# Build a Scikit Learn Model

In [9]:
# ! pip install scikit-learn

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/erkansirin78/datasets/master/Advertising.csv")
df.head()

,ID,TV,Radio,Newspaper,Sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


## Feature matrix

In [4]:
X = df.iloc[:, 1:-1].values
print(X.shape)
print(X[:3])

(200, 3)
[[230.1  37.8  69.2]
 [ 44.5  39.3  45.1]
 [ 17.2  45.9  69.3]]


## Output variable

In [5]:
y = df.iloc[:, -1]
print(y.shape)
print(y[:6])

(200,)
0    22.1
1    10.4
2     9.3
3    18.5
4    12.9
5     7.2
Name: Sales, dtype: float64


## Split data

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## Model trainig

In [11]:
from sklearn.ensemble import RandomForestRegressor
estimator = RandomForestRegressor(n_estimators=200)
estimator.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200)

## Model Test

In [12]:
y_pred = estimator.predict(X_test)
from sklearn.metrics import r2_score
r2 = r2_score(y_true=y_test, y_pred=y_pred)
print(f"R2: {r2}")

R2: 0.9825192010547583


## Save Model

In [13]:
import joblib
joblib.dump(estimator, "randomforest_with_advertising.pkl")

['randomforest_with_advertising.pkl']

## Read Model from Disk

In [14]:
estimator_loaded = joblib.load("randomforest_with_advertising.pkl")

## Some Manual predictions

In [15]:
X_manual_test = [[230.1,37.8,69.2]]
print("X_manual_test", X_manual_test)
prediction = estimator_loaded.predict(X_manual_test)
print("prediction", prediction)

X_manual_test [[230.1, 37.8, 69.2]]
prediction [22.0385]


# Scikit-Learn Model Deployment on PySpark

In [27]:
import findspark

findspark.init("/opt/spark/")
from pyspark.sql import DataFrame, Column, SparkSession, functions as F
from pyspark import SparkContext

In [19]:

spark = SparkSession.builder.appName("Spark ML").master("local[*]") \
.config("spark.driver.memory", "4g") \
.getOrCreate()
sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/29 11:34:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read dataset

In [20]:
from pyspark import SparkFiles
github_url="https://raw.githubusercontent.com/erkansirin78/datasets/master/Advertising.csv"
sc.addFile(github_url)
df = spark.read.csv(SparkFiles.get("Advertising.csv"),header= True, inferSchema=True)
df.show(5)

+---+-----+-----+---------+-----+
| ID|   TV|Radio|Newspaper|Sales|
+---+-----+-----+---------+-----+
|  1|230.1| 37.8|     69.2| 22.1|
|  2| 44.5| 39.3|     45.1| 10.4|
|  3| 17.2| 45.9|     69.3|  9.3|
|  4|151.5| 41.3|     58.5| 18.5|
|  5|180.8| 10.8|     58.4| 12.9|
+---+-----+-----+---------+-----+
only showing top 5 rows



## Prediction Function

In [25]:
def prediction(df:DataFrame, model, spark_context:SparkContext) -> Column:
    model = spark_context.broadcast(model)
    @F.udf('integer')
    def predict_data(*cols):
        return int(model.value.predict((cols,)))
    return predict_data(*df.columns)

## Prediction with Spark DF

In [28]:
df_pred = df.withColumn("PREDICTION", prediction(df=df.select('TV','Radio','Newspaper'), model=estimator_loaded, spark_context=sc))
df_pred.show(5)

+---+-----+-----+---------+-----+----------+
| ID|   TV|Radio|Newspaper|Sales|PREDICTION|
+---+-----+-----+---------+-----+----------+
|  1|230.1| 37.8|     69.2| 22.1|        22|
|  2| 44.5| 39.3|     45.1| 10.4|        10|
|  3| 17.2| 45.9|     69.3|  9.3|         8|
|  4|151.5| 41.3|     58.5| 18.5|        18|
|  5|180.8| 10.8|     58.4| 12.9|        13|
+---+-----+-----+---------+-----+----------+
only showing top 5 rows

